# 🎯 DipRadar — ML Training v3.1 (Colab)

Pipeline completo de treino dos modelos ML que correm no Railway.

**Outputs exportados para Railway:**
- `dip_models_v3.pkl` → modelo principal (joblib)
- `ml_report_v3.json` → métricas e metadados do treino
- `ml_training_base.parquet` → dataset v3.1 construído (ficheiro único)

**Schema de `ml_training_base.parquet` (ficheiro único — sem legados price/fund):**

| Grupo | Colunas |
|---|---|
| Identidade | `ticker`, `alert_date`, `sector` |
| Stage 0 — Macro | `macro_score`, `vix`, `spy_drawdown_5d`, `sector_drawdown_5d` |
| Stage 1 — Quality | `gross_margin`, `de_ratio`, `pe_vs_fair`, `analyst_upside`, `quality_score` |
| Stage 2 — Timing | `drop_pct_today`, `drawdown_52w`, `rsi_14`, `atr_ratio`, `volume_spike` |
| Stage 3 — Engineered | `rsi_oversold_strength`, `vix_regime`, `pe_attractive`, `drop_x_drawdown`, `vol_x_drop` |
| Stage 3b — Momentum | `return_1m`, `return_3m_pre`, `sector_relative`, `beta_60d` |
| Stage 3c — Dislocation | `quality_dislocation`, `peg_implicit`, `relative_drop`, `month_of_year` |
| Internos | `fcf_yield`, `revenue_growth`, `sector_alert_count_7d`, `days_since_52w_high` |
| Targets | `max_return_60d`, `max_drawdown_60d`, `spy_max_return_60d`, `alpha_60d` |

**Fluxo:**
1. Setup & autenticação GitHub
2. Clone do repo + install dependências
3. Download de dados de preços (Tiingo)
4. Build dataset v3.1
5. Walk-forward CV + seleção champion
6. Treino full + calibrador isotónico
7. Bundle + report
8. **Export**: download direto + push automático para GitHub

## ⚙️ Célula 1 — Setup: secrets, clone, install

In [ ]:
# ── Secrets do Colab (Google Colab → Secrets ícone 🔑)
# Necessário definir: GITHUB_TOKEN, TIINGO_API_KEY
import os
from google.colab import userdata

GITHUB_TOKEN  = userdata.get('GITHUB_TOKEN')   # PAT com scope repo
TIINGO_KEY    = userdata.get('TIINGO_API_KEY')
REPO_OWNER    = 'romeurf'
REPO_NAME     = 'DipRadar'
BRANCH        = 'main'

assert GITHUB_TOKEN, 'Falta GITHUB_TOKEN nos secrets do Colab'
assert TIINGO_KEY,   'Falta TIINGO_API_KEY nos secrets do Colab'
print('✅ Secrets carregados')

In [ ]:
import subprocess, sys

# Clone autenticado
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/{REPO_OWNER}/{REPO_NAME}.git'
REPO_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth=1', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print(f'✅ Repo em {REPO_DIR}')

In [ ]:
# Install dependências
!pip install -q -r requirements.txt
!pip install -q joblib scikit-learn lightgbm xgboost pandas pyarrow requests
print('✅ Dependências instaladas')

## 📥 Célula 2 — Carregar ml_training_base.parquet (ficheiro único)

In [ ]:
import pandas as pd
from pathlib import Path

REPO_PATH = Path(REPO_DIR)

# Único ficheiro de treino — contém as 34 features v3.1 + targets completos
# (ml_training_price.parquet e ml_training_fund.parquet foram removidos — eram artefactos v1/v2)
BASE_PARQUET = REPO_PATH / 'ml_training_base.parquet'

if BASE_PARQUET.exists():
    df_tmp = pd.read_parquet(BASE_PARQUET)
    print(f'✅ {BASE_PARQUET.name}: shape={df_tmp.shape}')
    print(f'   Período : {pd.to_datetime(df_tmp["alert_date"]).min().date()} → {pd.to_datetime(df_tmp["alert_date"]).max().date()}')
    print(f'   Tickers : {df_tmp["ticker"].nunique()}')
    print(f'   Colunas : {list(df_tmp.columns)}')
else:
    print('⚠️  ml_training_base.parquet não encontrado — será gerado na Célula 4')
    print('   (normal na primeira execução ou após reset do repo)')

## 🌐 Célula 3 — Download de preços via Tiingo

In [ ]:
import requests
from datetime import datetime, timedelta

os.environ['TIINGO_API_KEY'] = TIINGO_KEY

def tiingo_daily(ticker: str, start: str = '2018-01-01') -> pd.DataFrame:
    """Fetch OHLCV diário do Tiingo. Devolve DataFrame indexado por Date."""
    url = f'https://api.tiingo.com/tiingo/daily/{ticker}/prices'
    params = {
        'startDate': start,
        'endDate':   datetime.today().strftime('%Y-%m-%d'),
        'token':     TIINGO_KEY,
        'resampleFreq': 'daily',
        'columns':   'date,open,high,low,close,volume,adjClose'
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date').rename(columns={
        'open':'Open','high':'High','low':'Low',
        'close':'Close','volume':'Volume','adjClose':'AdjClose'
    })
    return df.sort_index()

# Buscar ETFs de referência
from ml_training.config import DEFAULT_ETF, SECTOR_ETF

etf_tickers = list(set([DEFAULT_ETF] + list(SECTOR_ETF.values())))
print(f'ETFs a descarregar: {etf_tickers}')

etf_cache: dict[str, pd.DataFrame] = {}
for etf in etf_tickers:
    try:
        df_etf = tiingo_daily(etf)
        if not df_etf.empty:
            etf_cache[etf] = df_etf
            print(f'  ✅ {etf}: {len(df_etf)} candles ({df_etf.index.min().date()} → {df_etf.index.max().date()})')
        else:
            print(f'  ⚠️  {etf}: sem dados')
    except Exception as e:
        print(f'  ❌ {etf}: {e}')

print(f'\n✅ ETF cache: {len(etf_cache)} ETFs carregados')

In [ ]:
# Buscar preços dos tickers no dataset base
# Se BASE_PARQUET existir, usa os tickers de lá; caso contrário falha com mensagem clara
from ml_training.data import load_base_dataset

if not BASE_PARQUET.exists():
    raise FileNotFoundError(
        'ml_training_base.parquet não encontrado.\n'
        'Na primeira execução absoluta (repo limpo), cria um parquet mínimo com:\n'
        '  pd.DataFrame({"ticker": [...], "alert_date": [...], "sector": [...]})'
        '.to_parquet(BASE_PARQUET, index=False)'
    )

base_df = load_base_dataset(BASE_PARQUET)
all_tickers = sorted(base_df['ticker'].unique())
print(f'Tickers no dataset base: {len(all_tickers)}')

price_cache: dict[str, pd.DataFrame] = {}
errors = []

for i, tk in enumerate(all_tickers):
    if (i + 1) % 50 == 0:
        print(f'  ... {i+1}/{len(all_tickers)}')
    try:
        df_p = tiingo_daily(tk)
        if not df_p.empty:
            price_cache[tk] = df_p
    except Exception as e:
        errors.append((tk, str(e)))

print(f'\n✅ Price cache: {len(price_cache)}/{len(all_tickers)} tickers')
if errors:
    print(f'⚠️  Erros ({len(errors)}): {errors[:5]} ...')

## 🏗️ Célula 4 — Build dataset v3.1

In [ ]:
from ml_training.config import HORIZON_DAYS
from ml_training.data import build_dataset_v31
from ml_features import FEATURE_COLUMNS, MOMENTUM_FEATURE_COLUMNS

NEW_FEATURES = ['relative_drop', 'sector_alert_count_7d', 'days_since_52w_high', 'month_of_year']
FEATURE_COLS_V31 = FEATURE_COLUMNS + MOMENTUM_FEATURE_COLUMNS + NEW_FEATURES

print(f'Features v3.1: {len(FEATURE_COLS_V31)} total')
print(f'  Base:     {len(FEATURE_COLUMNS)}')
print(f'  Momentum: {len(MOMENTUM_FEATURE_COLUMNS)}')
print(f'  New v3.1: {len(NEW_FEATURES)}')
print(f'Horizon: {HORIZON_DAYS} dias')
print('\nA construir dataset v3.1 (pode demorar alguns minutos)...')

df_v31, skipped = build_dataset_v31(
    base_df=base_df,
    price_cache=price_cache,
    etf_cache=etf_cache,
    feature_cols_v31=FEATURE_COLS_V31,
    horizon_days=HORIZON_DAYS,
)

print(f'\n✅ Dataset v3.1: {df_v31.shape}')
print(f'   Skipped: {skipped}')
print(f'   Período: {df_v31["alert_date"].min().date()} → {df_v31["alert_date"].max().date()}')
df_v31.head(3)

## 📊 Célula 5 — Walk-forward CV

In [ ]:
from ml_training.models import get_model_configs
from ml_training.train import run_walk_forward_cv, summarize_results, select_champion

# Hiperparâmetros do CV
N_FOLDS    = 8
PURGE_DAYS = 30

model_configs = get_model_configs(FEATURE_COLS_V31)
print(f'Modelos candidatos: {list(model_configs.keys())}')
print(f'CV: {N_FOLDS} folds, purge={PURGE_DAYS}d')
print('\nA correr walk-forward CV...')

import logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

results, oof_pred, fold_specs = run_walk_forward_cv(
    df_v31=df_v31,
    model_configs=model_configs,
    n_folds=N_FOLDS,
    purge_days=PURGE_DAYS,
)

summary = summarize_results(results)
print('\n=== Resumo CV ===')
print(summary.to_string(index=False))

In [ ]:
# Selecionar champion
champion_name, champion_row = select_champion(summary)
print(f'\n🏆 Champion: {champion_name}')
print(f'   rho_alpha_mean : {champion_row["rho_alpha_mean"]:.4f}')
print(f'   topk_pnl_mean  : {champion_row["topk_pnl_mean"]:.4f}')
print(f'   n_folds        : {int(champion_row["n_folds"])}')

## 🎓 Célula 6 — Treino full + calibrador

In [ ]:
import numpy as np
from ml_training.train import train_full_champion, fit_isotonic_calibrator

champion_cfg = model_configs[champion_name]

print('A treinar champion no dataset completo...')
champ_alpha, champ_down, feats_used, n_train = train_full_champion(df_v31, champion_cfg)
print(f'✅ Treinado: {n_train} amostras, {len(feats_used)} features')

# Calibrador isotónico (OOF)
oof_champion = oof_pred[champion_name]
alpha_true   = df_v31['alpha_60d'].values

iso_model, brier_oof, n_oof = fit_isotonic_calibrator(
    oof_pred_champion=oof_champion,
    alpha_true=alpha_true,
    alpha_threshold=0.05,
)
print(f'✅ Calibrador OOF: brier={brier_oof:.4f}, n_oof={n_oof}')

# Win-rate alpha (para o report)
valid_mask = np.isfinite(oof_champion)
win_rate_alpha = float((alpha_true[valid_mask] > 0.05).mean()) if valid_mask.sum() > 0 else 0.0
print(f'   win_rate_alpha (>5%): {win_rate_alpha:.3f}')

## 📦 Célula 7 — Bundle + Report

In [ ]:
from datetime import datetime
from ml_training.bundle import DipModelsV3, save_bundle, build_report, save_report
from ml_training.config import HORIZON_DAYS

# Recuperar métricas do CV do champion
champ_hist = results[champion_name]
rho_alpha_vals = [h['rho_alpha'] for h in champ_hist if np.isfinite(h['rho_alpha'])]
rho_down_vals  = [h['rho_down']  for h in champ_hist if np.isfinite(h['rho_down'])]
pnl_vals       = [h['topk_pnl']  for h in champ_hist if np.isfinite(h['topk_pnl'])]

from ml_features import MOMENTUM_FEATURE_COLUMNS

bundle = DipModelsV3(
    model_up         = champ_alpha,
    model_down       = champ_down,
    feature_cols     = feats_used,
    score_calibrator = iso_model,
    n_train_samples  = n_train,
    train_date       = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
    champion_name    = champion_name,
    schema_version   = 3,
    momentum_feats   = MOMENTUM_FEATURE_COLUMNS,
    rho_mean         = float(np.mean(rho_alpha_vals)) if rho_alpha_vals else None,
    rho_alpha        = float(np.mean(rho_alpha_vals)) if rho_alpha_vals else None,
    rho_down         = float(np.mean(rho_down_vals))  if rho_down_vals  else None,
    topk_pnl         = float(np.mean(pnl_vals))       if pnl_vals       else None,
    fold_metrics     = champ_hist,
)

# Caminhos locais (no Colab)
PKL_PATH    = Path('/content/dip_models_v3.pkl')
REPORT_PATH = Path('/content/ml_report_v3.json')

save_bundle(bundle, PKL_PATH)
print(f'✅ Bundle guardado: {PKL_PATH} ({PKL_PATH.stat().st_size/1024:.1f} KB)')

report = build_report(
    bundle         = bundle,
    summary_df     = summary,
    brier_oof      = brier_oof,
    win_rate_alpha = win_rate_alpha,
    n_folds_used   = len(champ_hist),
    purge_days     = PURGE_DAYS,
    horizon_days   = HORIZON_DAYS,
    new_features   = NEW_FEATURES,
)
save_report(report, REPORT_PATH)
print(f'✅ Report guardado: {REPORT_PATH}')
print(f"   Champion : {report['champion']}")
print(f"   rho_alpha: {report['metrics']['rho_alpha_mean']}")
print(f"   topk_pnl : {report['metrics']['topk_pnl_mean']}")
print(f"   brier_oof: {report['metrics']['brier_oof']}")

## 💾 Célula 8 — Guardar ml_training_base.parquet atualizado

Guarda o dataset v3.1 completo como `ml_training_base.parquet` no Colab.
Este ficheiro contém as 34 features + targets — é o único ficheiro de treino necessário.

In [ ]:
PARQUET_PATH = Path('/content/ml_training_base.parquet')
df_v31.to_parquet(PARQUET_PATH, index=False)
print(f'✅ ml_training_base.parquet guardado: {PARQUET_PATH}')
print(f'   Tamanho : {PARQUET_PATH.stat().st_size/1024:.0f} KB')
print(f'   Linhas  : {len(df_v31)}')
print(f'   Colunas : {len(df_v31.columns)} → {list(df_v31.columns)}')

## 📤 Célula 9 — Export: Download direto do Colab

**Opção A (mais simples):** Download manual dos 3 ficheiros para o teu PC, depois faz drag-and-drop no Railway ou push manual.


In [ ]:
from google.colab import files

print('A iniciar downloads...')
files.download(str(PKL_PATH))
files.download(str(REPORT_PATH))
files.download(str(PARQUET_PATH))
print('✅ 3 ficheiros prontos para download:')
print('   📦 dip_models_v3.pkl')
print('   📊 ml_report_v3.json')
print('   🗃️  ml_training_base.parquet')

## 🚀 Célula 10 — Export: Push automático para GitHub

**Opção B (recomendado):** Push direto dos 3 ficheiros para o repo.
O Railway deteta o push e redeploy automaticamente com os novos modelos.


In [ ]:
import base64
import json as _json
import requests as _req
from datetime import datetime as _dt

GH_API = 'https://api.github.com'
HEADERS = {
    'Authorization': f'token {GITHUB_TOKEN}',
    'Accept':        'application/vnd.github.v3+json',
}

def gh_get_sha(filepath: str) -> str | None:
    """Obtém o SHA do ficheiro atual no repo (necessário para update)."""
    url = f'{GH_API}/repos/{REPO_OWNER}/{REPO_NAME}/contents/{filepath}'
    r = _req.get(url, headers=HEADERS, params={'ref': BRANCH})
    if r.status_code == 200:
        return r.json().get('sha')
    return None  # ficheiro ainda não existe

def gh_push_file(filepath: str, local_path: Path, commit_msg: str) -> None:
    """Faz create-or-update de um ficheiro via GitHub API."""
    content_b64 = base64.b64encode(local_path.read_bytes()).decode()
    sha = gh_get_sha(filepath)
    url = f'{GH_API}/repos/{REPO_OWNER}/{REPO_NAME}/contents/{filepath}'
    payload: dict = {
        'message': commit_msg,
        'content': content_b64,
        'branch':  BRANCH,
    }
    if sha:
        payload['sha'] = sha
    r = _req.put(url, headers=HEADERS, data=_json.dumps(payload))
    if r.status_code in (200, 201):
        verb = 'Atualizado' if sha else 'Criado'
        print(f'  ✅ {verb}: {filepath}')
    else:
        print(f'  ❌ Erro em {filepath}: {r.status_code} — {r.text[:200]}')

ts = _dt.utcnow().strftime('%Y-%m-%d %H:%M UTC')
commit_msg = f'chore(ml): retrain v3.1 [{ts}] champion={champion_name} rho={bundle.rho_alpha:.4f}'

print(f'Push para {REPO_OWNER}/{REPO_NAME}@{BRANCH}')
print(f'Commit: {commit_msg}\n')

gh_push_file('dip_models_v3.pkl',          PKL_PATH,    commit_msg)
gh_push_file('ml_report_v3.json',          REPORT_PATH, commit_msg)
gh_push_file('ml_training_base.parquet',   PARQUET_PATH, commit_msg)

print('\n🚀 Push concluído — Railway irá redeploy automaticamente.')